In [1]:
import pandas as pd
import scipy as sp
import numpy as np
from lightfm import cross_validation
from lightfm.evaluation import precision_at_k
from lightfm.evaluation import recall_at_k
from lightfm.evaluation import auc_score
from lightfm import LightFM
import matplotlib.pyplot as plt
import pickle

In [2]:
#load data
data = []
with open("/home/nasi4/proj/GWAS/UKBB_201807/results/all.UKBB.hg38.sig.phenotype.bed")as f:
    for line in f:
        data.append(line.strip().split())

In [3]:
#combine label elements
for row in data:
    row.append(' '.join(row[5:]))
    del row[5:-1]

In [4]:
#create dataframe
df = pd.DataFrame(data, columns = ['Chromosome', 'p1', 'p2', 'Reference SNP cluster ID', 'p-value', 'Trait'])

In [5]:
#calculating the negative log of the p-value, setting infinite values to the max non-infinite value observed, and
#normalizing by max -log(p-value) score
pval_mlog = -np.log(list(map(float, list(df['p-value']))))

inf_mask = np.ma.array(pval_mlog, mask=~np.isfinite(pval_mlog))
max_val = np.ma.argmax(inf_mask, fill_value=-np.inf)
print(pval_mlog[max_val])

for ind in range(len(pval_mlog)):
    if pval_mlog[ind] > pval_mlog[max_val]:
        pval_mlog[ind] = pval_mlog[max_val]
        
df['pval_mlog'] = pval_mlog/pval_mlog[max_val]

/tmp/ipykernel_170573/734574533.py:3: RuntimeWarning: divide by zero encountered in log
  pval_mlog = -np.log(list(map(float, list(df['p-value']))))


743.7469247408213


In [95]:
#creating df with SNP, trait, and p-value
ts_df = df[['Reference SNP cluster ID', 'Trait', 'pval_mlog']]
ts_df.head()

,Reference SNP cluster ID,Trait,pval_mlog
0,rs547522712,Diagnoses - main ICD10: L98 Other disorders of...,0.061532
1,rs547522712,Other disorders of skin and subcutaneous tissu...,0.061532
2,rs547522712,#Spondylopathies,0.058265
3,rs547522712,Diagnoses - main ICD10: M48 Other spondylopathies,0.122894
4,rs547522712,Other and unspecified disorders of skin and su...,0.066955


In [89]:
#finding unique traits and SNPs
SNP_values = np.unique(ts_df['Reference SNP cluster ID'].values)
trait_values = np.unique(ts_df['Trait'].values)

In [96]:
ts = ts_df.groupby(['Reference SNP cluster ID', 'Trait']).mean()
ts = ts.reset_index()

In [107]:
print(len(trait_values))
print(len(SNP_values))

3815
2090236


In [12]:
# ts[(ts['Trait']=='3mm strong meridian (right)') & (ts['Reference SNP cluster ID']=='rs570860110')]

,Reference SNP cluster ID,Trait,pval_mlog
1009286,rs570860110,3mm strong meridian (right),0.033397
1009292,rs570860110,3mm strong meridian (right),0.032463
1009300,rs570860110,3mm strong meridian (right),0.035064
1009308,rs570860110,3mm strong meridian (right),0.034136


In [101]:
# ts[(ts['Trait']=='3mm strong meridian (right)') & (ts['Reference SNP cluster ID']=='rs570860110')]

,Reference SNP cluster ID,Trait,pval_mlog
7872741,rs570860110,3mm strong meridian (right),0.033765


In [102]:
ts['trait_ind'] = ts.groupby(['Trait']).ngroup()
ts['snp_ind'] = ts.groupby(['Reference SNP cluster ID']).ngroup()

In [105]:
ts

,Reference SNP cluster ID,Trait,pval_mlog,trait_ind,snp_ind
0,10:100014847_CT_C,Impedance of arm (left),0.038691,1436,0
1,10:100014847_CT_C,Impedance of arm (right),0.029557,1437,0
2,10:100014847_CT_C,Impedance of leg (left),0.027193,1438,0
3,10:100014847_CT_C,Impedance of leg (right),0.031024,1439,0
4,10:100014847_CT_C,Impedance of whole body,0.035546,1440,0
...,...,...,...,...,...
11397252,rs9999952,Whole body fat-free mass,0.024137,3765,2090232
11397253,rs9999952,Whole body water mass,0.024719,3766,2090232
11397254,rs9999960,Sitting height,0.024805,2896,2090233
11397255,rs9999976,Alcohol intake frequency.,0.027955,100,2090234


In [106]:
columns = ts['snp_ind'].to_list()
rows = ts['trait_ind'].to_list()
pval_mlog_data = ts['pval_mlog'].to_list()

In [110]:
data_coo = sp.sparse.coo_matrix((pval_mlog_data, (rows, columns)), shape=(len(trait_values), len(SNP_values)))

In [111]:
print(data_coo.max())

1.0


In [112]:
#splitting data into train and test
(train, test) = cross_validation.random_train_test_split(data_coo, test_percentage=0.2, random_state=None)

In [113]:
data = {'train': train, 'test': test}

In [114]:
#creating model
gwas_model = LightFM(no_components=10, loss='warp')

In [115]:
#training model
gwas_model.fit(data['train'], epochs=30, num_threads=2)

In [116]:
pickle_out_data = open('ts_data_CT20211001.pickle', 'wb')
pickle.dump(data, pickle_out_data)
pickle_out_data.close()

pickle_out_model = open('ts_model_CT20211001.pickle', 'wb')
pickle.dump(gwas_model, pickle_out_model)
pickle_out_model.close()

In [ ]:
#function to recommend genes for a given disease. Takes in the model, data, and a list of
#diseases for genes to be recommend for. For each disease, prints out known associated genes
#and a list of recommended genes. Function modeled after tutorial 3, 'Recommendation Systems
#- Learn Python for Data Science' on LightFM github. 

def sample_recommendation(model, data, diseases, recommend_num=5, full_data=gd_coo):
    
    #number of diseases and genes in training dataset
    n_diseases, n_genes = data['train'].shape
    
    #generate gene recommendations for each disease
    for disease in diseases:
        disease_id = df[df['diseaseName'] == disease]['diseaseId'].tolist()[0]
        disease_id_num = list(diseaseId_values).index(disease_id)
        #genes known to relate to the disease
        known_positives = data['gene_ids'][data['train'].tocsr()[disease_id_num].indices]
#         known_positives = data['gene_ids'][np.argsort(-gd_coo.todense()[disease_id_num, :])]
        known_ranked = list(np.squeeze(np.asarray(np.argsort(-gd_coo.todense()[disease_id_num, :]))))
#         print(known_ranked)
        
        #genes model predicts that relate to disease
        scores = model.predict(disease_id_num, np.arange(n_genes))
#         print(np.argsort(-scores))
#         print(max(np.argsort(-scores)))
        #rank from most related to least
        top_genes = data['gene_ids'][np.argsort(-scores)]
        
        #print results
        print('Disease: ' + disease + ' (' + str(disease_id_num) +')')
        print('   Known Positives:')
        
        known_pos = []
        gd_dense = gd_coo.todense()
        for gene in known_ranked[:5]:
            association_score = gd_dense[disease_id_num, gene]
#             association_score = df[(df['geneId'] == gene) & (df['diseaseName']==disease)]['score'].tolist()[0]
            if association_score == 0:
                break
            geneId = data['gene_ids'][gene]
            x = df[df['geneId'] == geneId]['geneSymbol'].tolist()[0]
            known_pos.append(x)
            print('      %s (%s)' %(x, association_score))
        
        print()
        print('   Recommended:')
        count = 0
        for gene in top_genes[:]:
            x = df[df['geneId'] == gene]['geneSymbol'].tolist()[0]
            if x not in known_pos:
                print('      %s' % x)
                count += 1
            if count == recommend_num:
                break
                
        print()
            
sample_recommendation(model, data, ['Lung Neoplasms'])
sample_recommendation(model_scaled, data_scaled, ['Lung Neoplasms'], full_data=gd_scaled)

In [ ]:
n_traits, n_snp = data['train'].shape
scores = model.predict(15, np.arange(n_SNPs))
top_genes = data['gene_ids'][np.argsort(-scores)]

In [19]:
df[(df['Reference SNP cluster ID']=='rs570860110')&(df['Trait']=='3mm strong meridian (right)')]

,Chromosome,p1,p2,Reference SNP cluster ID,p-value,Trait,pval_mlog
1009286,chr1,202817700,202817701,rs570860110,1.63190e-11,3mm strong meridian (right),0.033397
1009292,chr1,202817700,202817701,rs570860110,3.26722e-11,3mm strong meridian (right),0.032463
1009300,chr1,202817700,202817701,rs570860110,4.72082e-12,3mm strong meridian (right),0.035064
1009308,chr1,202817700,202817701,rs570860110,9.41552e-12,3mm strong meridian (right),0.034136


In [99]:
# counter = 0
# for trait in trait_values:
# #     print(trait)
#     trait_df = ts_df[ts_df['Trait']==trait].set_index('Reference SNP cluster ID')
#     duplicate_list = np.unique(list(trait_df[trait_df.index.duplicated()].index))
#     if len(duplicate_list) > 0:
#         entry_list = np.array([])
#         for duplicate in duplicate_list:
# #             print(duplicate)
#             ave = np.mean(trait_df.loc[duplicate]['pval_mlog'])
#             trait_df = trait_df.drop(duplicate)
#             np.append(entry_list, [trait, ave, duplicate], axis=0)
#         entry = pd.DataFrame(list(entry_list), columns=['Trait', 'pval_mlog', 'Reference SNP cluster ID'])
#         entry = entry.set_index('Reference SNP cluster ID')
# #         print('entry done')
#         trait_df = trait_df.append(entry)
# #         print('df done')
#     ts_matrix.loc[trait] = trait_df['pval_mlog']
# #     print('main df done')
#     if counter%50 == 0:
#         print('Trait %s done' %(counter))
#     counter+=1

# ts_np = ts_matrix.to_numpy()
# ts_np = np.array(ts_np, dtype=float)
# ts_np[np.isnan(ts_np)] = 0
# ts_coo = sp.sparse.coo_matrix(ts_np)
# print(ts_coo)